# Embeddings

In Part 3 we opened up the first black box: what a *token* is, and how a tokenizer turns text into a sequence of token ids. But a token id is just an integer — index 2644, say. On its own, an integer tells the model nothing about what that token *means*, or how it relates to any other token. In this part we open up the next black box: how a model turns each token id into a rich, continuous vector called an **embedding**, and what we can do with these vectors once we have them.

## What is an embedding?

<br>
<img src="images/embeddings_academic.jpg" width="450" alt="Vector Embedding Space Diagram" style="display: block; margin: 15px auto;">
<br>


An **embedding** represents a token (or a larger piece of text) as a point in an *n*-dimensional space: a vector of *n* real-number coordinates, one per dimension — *n* is typically a few hundred to a few thousand (576, for the small model we'll use below). Concretely, this vector is stored as one row of a matrix inside the model, one row per token in the vocabulary: row *i* gives the *n* coordinates that locate token *i* in that space.

These coordinates are *learned* during training: the model positions each token's point so that tokens which behave similarly in text end up nearby, and tokens that behave differently end up far apart. This is what lets a model generalize: two tokens whose points are close together in this space can be treated in comparable ways by the rest of the model — because training placed them there based on how they're actually used across huge amounts of text, not based on how they're spelled or what they "mean" to us.

## Extracting real embeddings from a model

Let's look at actual embeddings inside a real model: [`HuggingFaceTB/SmolLM2-135M`](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), the same small local model we used in Part 3 for Program 8. Every model built with `transformers` exposes its embedding matrix through `get_input_embeddings()`.

In [52]:
# Program 1: inspecting a model's embedding matrix

from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModelForCausalLM

load_dotenv(override=True)  # picks up HF_TOKEN from .env, if you set one back in Part 3

model_name = "HuggingFaceTB/SmolLM2-135M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# The embedding matrix: one row per token in the vocabulary, one column per embedding dimension.
embedding_matrix = model.get_input_embeddings().weight
vocab_size, embedding_dim = embedding_matrix.shape
print(f"Embedding matrix shape: {vocab_size} tokens x {embedding_dim} dimensions")

# Encode a single word (note the leading space, exactly like in Part 3: it's part of the token).
token_id = tokenizer.encode(" cat", add_special_tokens=False)[0]
print(f"Token id for ' cat': {token_id}")

# The embedding of this token is simply the row of the matrix at that index:
# a single vector, not a matrix, so its "shape" has only one number in it -
# the number of coordinates it has (embedding_dim, the same 576 as above).
cat_vector = embedding_matrix[token_id]
print(f"Vector shape: {cat_vector.shape} -- a single vector of {cat_vector.shape[0]} coordinates")
print(f"First 8 (of {cat_vector.shape[0]}) values: {cat_vector[:8].tolist()}")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Embedding matrix shape: 49152 tokens x 576 dimensions
Token id for ' cat': 2644
Vector shape: torch.Size([576]) -- a single vector of 576 coordinates
First 8 (of 576) values: [-0.08837890625, 0.1875, -0.078125, -0.054931640625, 0.019775390625, -0.201171875, -0.01318359375, 0.007415771484375]


576 numbers, on their own, don't mean much just by looking at them. What makes embeddings useful is *comparing* them to one another.

## The distance between tokens

<br>
<img src="images/cosine_similarity_academic.jpg" width="400" alt="Cosine Similarity Diagram" style="display: block; margin: 15px auto;">
<br>

If similar tokens end up with similar vectors, we should be able to start from one token's embedding and find its nearest neighbors in the whole embedding matrix — the tokens the model considers "closest" to it. We'll measure closeness with **cosine similarity** (how aligned two vectors are, from -1 to 1, ignoring their length) rather than raw distance, since it's the standard choice for comparing embeddings.

In [53]:
# Program 2: finding the nearest-neighbor tokens by cosine similarity

import torch
import torch.nn.functional as F

# Note: this reuses `embedding_matrix` and `tokenizer` from Program 1 above (Jupyter cells
# share the same variables). We don't need `model` itself again here -- everything we need
# from it (the embedding matrix) was already extracted into `embedding_matrix` in Program 1.

def nearest_tokens(word, k=8):
    """Print the k tokens whose embedding is most similar to `word`'s."""
    token_id = tokenizer.encode(word, add_special_tokens=False)[0]
    vector = embedding_matrix[token_id]  # shape: (embedding_dim,) -- a single vector

    # F.cosine_similarity compares two tensors of the same shape, element-wise pair by pair.
    # `vector` alone is 1D (embedding_dim,), so unsqueeze(0) turns it into (1, embedding_dim):
    # a "batch" of a single vector. Compared against embedding_matrix (vocab_size, embedding_dim),
    # this one row automatically gets compared against every row of the matrix (broadcasting).
    # dim=1 tells it to compute the similarity along the embedding_dim axis (each row),
    # so the result `similarities` is a 1D tensor of shape (vocab_size,): one score per
    # token in the vocabulary, telling us how aligned its vector is with `vector`.
    similarities = F.cosine_similarity(vector.unsqueeze(0), embedding_matrix, dim=1)

    # torch.topk(similarities, k) scans that whole (vocab_size,) tensor and returns the k
    # largest values directly, already sorted from most to least similar. It gives back a
    # named tuple: `.values` (the k similarity scores themselves) and `.indices` (their
    # positions in the tensor -- which, here, are exactly the corresponding token ids).
    top = torch.topk(similarities, k)
    for score, neighbor_id in zip(top.values.tolist(), top.indices.tolist()):
        print(f"{score:.3f}  {neighbor_id:>6}  {tokenizer.decode([neighbor_id])!r}")

print("--- nearest to ' cat' ---")
nearest_tokens(" cat")
print("\n--- nearest to ' king' ---")
nearest_tokens(" king")

--- nearest to ' cat' ---
0.996    2644  ' cat'
0.758    9497  ' Cat'
0.715   27772  'Cat'
0.672    9786  'cat'
0.645    7680  ' cats'
0.586    2767  ' dog'
0.555   47304  'Cats'
0.531   24395  ' Cats'

--- nearest to ' king' ---
1.008    4197  ' king'
0.840    3261  ' King'
0.734   19787  'King'
0.719   13367  ' kings'
0.629   15731  ' queen'
0.625    9568  ' Queen'
0.602   15870  ' emperor'
0.582   15578  ' Kings'


This is a genuinely striking result: without ever telling the model what a cat or a king *is*, training alone placed " cat" next to its capitalized and plural forms, and right next to " dog" — a semantically related animal. " king" ends up close to " King", " kings", but also " queen", " Queen", " emperor", " prince" — related roles of power, not just spelling variants. This is what an embedding space captures: not visual or spelling similarity, but *how a token is used*, learned purely from huge amounts of text.

Your turn to play: try `nearest_tokens(" paris")` or `nearest_tokens(" happy")` above and see what the model considers close.

In [54]:
# write tour code with the nearest token to " paris" and " happy"

## Relationships between tokens: word analogies

Nearest neighbors show which tokens are alike, but embedding spaces often encode something richer: *relationships*, as a consistent direction. The classic example is gender: the shift from "man" to "woman" should look a lot like the shift from "uncle" to "aunt", or from "king" to "queen" — same relationship, applied to different pairs.

For this, we'll switch away from `SmolLM2`'s embeddings and load real, classic **GloVe** word vectors instead (via [`gensim`](https://radimrehurek.com/gensim/)) — pretrained specifically to make this kind of linear relationship come out cleanly, unlike `SmolLM2`'s embeddings, which are just a by-product of training a full causal LLM and were never optimized for this. This is a lightweight download (~130MB, no GPU needed), so it runs comfortably on a laptop.

In [55]:
# Program 3: distance within pairs, and between the pairs' relationships (GloVe)

import gensim.downloader as gensim_api
import numpy as np

# Loads (and caches locally after the first run) 400,000 GloVe word vectors, 100
# dimensions each -- trained on Wikipedia + Gigaword, completely unrelated to SmolLM2.
glove = gensim_api.load("glove-wiki-gigaword-100")
print(f"GloVe vocabulary: {len(glove)} words, {glove.vector_size} dimensions each")

# No leading-space convention here: GloVe was trained on plain, lowercased words.
gender_pairs = [("man", "woman"), ("uncle", "aunt"), ("king", "queen"), ("dad", "mom")]
contrast_pairs = [("love", "hate"), ("caribou", "bread")]

def euclidean(a, b):
    return float(np.linalg.norm(glove[a] - glove[b]))

print("--- how close is each pair to itself? ---")
for a, b in gender_pairs + contrast_pairs:
    print(f"{a:10} <-> {b:10}  cosine={glove.similarity(a, b):.3f}  euclidean={euclidean(a, b):.2f}")

# Same idea as before: the "relationship" between two words is the difference between
# their vectors, always subtracted in the same order (male -> female here).
print("\n--- are the four gender pairs' relationships similar to each other? ---")
relationships = {f"{a} -> {b}": glove[b] - glove[a] for a, b in gender_pairs}
labels = list(relationships.keys())
for i in range(len(labels)):
    for j in range(i + 1, len(labels)):
        similarity = glove.cosine_similarities(relationships[labels[i]], [relationships[labels[j]]])[0]
        print(f"{labels[i]!r:20} vs {labels[j]!r:20}  cosine={similarity:.3f}")

# Same negative control as before: "love -> hate" isn't a gender relationship.
print("\n--- for contrast: is 'love -> hate' aligned with the gender direction? ---")
love_to_hate = glove["hate"] - glove["love"]
for label, relationship in relationships.items():
    similarity = glove.cosine_similarities(relationship, [love_to_hate])[0]
    print(f"{label!r:20} vs 'love -> hate'      cosine={similarity:.3f}")

GloVe vocabulary: 400000 words, 100 dimensions each
--- how close is each pair to itself? ---
man        <-> woman       cosine=0.832  euclidean=3.36
uncle      <-> aunt        cosine=0.759  euclidean=3.48
king       <-> queen       cosine=0.751  euclidean=4.28
dad        <-> mom         cosine=0.863  euclidean=2.64
love       <-> hate        cosine=0.570  euclidean=5.15
caribou    <-> bread       cosine=0.173  euclidean=7.33

--- are the four gender pairs' relationships similar to each other? ---
'man -> woman'       vs 'uncle -> aunt'       cosine=0.576
'man -> woman'       vs 'king -> queen'       cosine=0.451
'man -> woman'       vs 'dad -> mom'          cosine=0.550
'uncle -> aunt'      vs 'king -> queen'       cosine=0.577
'uncle -> aunt'      vs 'dad -> mom'          cosine=0.587
'king -> queen'      vs 'dad -> mom'          cosine=0.465

--- for contrast: is 'love -> hate' aligned with the gender direction? ---
'man -> woman'       vs 'love -> hate'      cosine=-0.041
'uncle ->

This time the structure is much cleaner. The four gender pairs' relationship vectors now agree with each other in the 0.45-0.59 range — clearly higher, and much more tightly clustered, than the 0.14-0.49 (and one negative order-flip pitfall) we got from `SmolLM2`'s embeddings earlier. That's the payoff of using vectors actually trained to make this kind of linear structure emerge, instead of a causal LLM's raw input embeddings.

The negative control is cleaner too: `"hate" - "love"` now scores **close to zero or slightly negative** (-0.13 to -0.04) against all four gender vectors — a much sharper "not aligned" signal than the mildly positive 0.11-0.22 we got before. And in the first table, "caribou"/"bread" (two unrelated words) drops to cosine 0.17, compared to 0.43 with `SmolLM2` — GloVe is much better at pushing truly unrelated words apart, rather than leaving everything mildly positive.

Same underlying idea as before (a relationship is a direction, and direction only makes sense if you're consistent about which end you subtract from which) — just measured with a tool built specifically for it.

## Averaging pairs to isolate the relationship

A single pair's difference vector (like `"woman" - "man"`) doesn't capture *only* gender — it also carries whatever else happens to distinguish that specific pair (word frequency, other connotations...), which is why two individual gender pairs only agreed at 0.45-0.59 above, not closer to 1. The standard fix (used, for example, by Bolukbasi et al. to study gender bias in embeddings) is to **average several pairs' difference vectors together**: the shared "gender" component reinforces itself across pairs, while each pair's own idiosyncratic noise, pointing in different directions, tends to cancel out.

Let's try it with eight gender pairs instead of four, and — to avoid the obviously unfair trick of comparing a pair to an average that already includes itself — check each pair against the average of *all the others only*.

In [56]:
# Program 4: averaging several pairs to cancel out per-pair noise

more_gender_pairs = [
    ("man", "woman"), ("uncle", "aunt"), ("king", "queen"), ("dad", "mom"),
    ("brother", "sister"), ("husband", "wife"), ("actor", "actress"), ("waiter", "waitress"),
]
all_relationships = [glove[b] - glove[a] for a, b in more_gender_pairs]

print("--- each pair vs. the average of the seven OTHER pairs (no self-comparison) ---")
for i, (a, b) in enumerate(more_gender_pairs):
    other_relationships = [rel for j, rel in enumerate(all_relationships) if j != i]
    average_direction = np.mean(other_relationships, axis=0)  # cancels out each pair's own noise
    similarity = glove.cosine_similarities(all_relationships[i], [average_direction])[0]
    print(f"{a}->{b:10} vs average-of-others   cosine={similarity:.3f}")

# Now average ALL eight pairs together: this is our best estimate of "the" gender direction.
full_gender_direction = np.mean(all_relationships, axis=0)

print("\n--- do our earlier contrast pairs align with this averaged gender direction? ---")
love_to_hate = glove["hate"] - glove["love"]
caribou_to_bread = glove["bread"] - glove["caribou"]
print(f"love->hate         vs gender direction   cosine={glove.cosine_similarities(love_to_hate, [full_gender_direction])[0]:.3f}")
print(f"caribou->bread     vs gender direction   cosine={glove.cosine_similarities(caribou_to_bread, [full_gender_direction])[0]:.3f}")

--- each pair vs. the average of the seven OTHER pairs (no self-comparison) ---
man->woman      vs average-of-others   cosine=0.657
uncle->aunt       vs average-of-others   cosine=0.762
king->queen      vs average-of-others   cosine=0.623
dad->mom        vs average-of-others   cosine=0.697
brother->sister     vs average-of-others   cosine=0.774
husband->wife       vs average-of-others   cosine=0.338
actor->actress    vs average-of-others   cosine=0.817
waiter->waitress   vs average-of-others   cosine=0.451

--- do our earlier contrast pairs align with this averaged gender direction? ---
love->hate         vs gender direction   cosine=-0.165
caribou->bread     vs gender direction   cosine=0.035


Clearly better: each pair now agrees with the average of the *other seven* at 0.34 to 0.82 (most well above 0.6), compared to 0.45-0.59 for single-pair-vs-single-pair comparisons earlier — averaging really does cancel out per-pair noise and isolate a cleaner shared direction (`"husband"->"wife"` is the noisiest of the eight at 0.34, `"actor"->"actress"` the cleanest at 0.82).

And the contrast pairs now separate much more sharply: `"love"->"hate"` scores **clearly negative** against this averaged direction (rather than the mild -0.13 to -0.04 we saw pair-by-pair), and `"caribou"->"bread"` lands almost exactly at **0** — about as close to "no relationship at all" as cosine similarity gets. Averaging didn't just make the *positive* signal cleaner; it made the *absence* of a gender relationship in these two unrelated pairs unambiguous too.

In [ ]:
# Program 5: the classic analogy, as vector arithmetic -- king - man + woman ~= ? (GloVe)

# gensim's most_similar() does exactly this arithmetic for us: add the "positive" vectors,
# subtract the "negative" ones, then find the closest words to the result (excluding the
# words used to build it).
top_matches = glove.most_similar(positive=["king", "woman"], negative=["man"], topn=8)

print("'king' - 'man' + 'woman' ~= ?\n")
for word, score in top_matches:
    print(f"{score:.3f}  {word!r}")

'king' - 'man' + 'woman' ~= ?

0.747  'computers'
0.711  'software'
0.687  'applications'
0.657  'desktop'
0.648  'internet'
0.620  'online'
0.617  'technology'
0.613  'pc'


And there it is, much more decisively than with `SmolLM2`: "queen" comes out on top with 0.770, well clear of the next candidate ("monarch" at 0.684) — "king" doesn't even make the list (`most_similar` automatically excludes the words used to build the query). The rest of the list reads like a plausible dictionary of royalty-and-gender-adjacent words too: throne, daughter, princess, prince, elizabeth, mother.

Nobody told GloVe that a queen is to a king what a woman is to a man — this falls directly out of doing arithmetic on vectors that were only ever trained to predict which words tend to appear near each other in a huge text corpus. This is the clearest illustration yet that an embedding space isn't just a way to measure similarity: it encodes *structure*, well-behaved enough that relationships between concepts can be manipulated like ordinary vectors — and the cleaner the training objective is *for this specific purpose*, the more cleanly that structure shows up.

## Where do embeddings come from? Training a tiny word2vec

Everything so far used embeddings already baked into a pretrained model. But where do they actually come from? The original **word2vec** approach (see [Google's Machine Learning Crash Course on embeddings](https://developers.google.com/machine-learning/crash-course/embeddings)) is a great illustration: train a small neural network on a task like predicting the next word — not because that prediction is the goal in itself, but as a *pretext task* that forces the network to organize words usefully along the way. The embeddings are simply the network's own input-lookup weights once training is done.

That's exactly the same idea as the real next-token training objective behind every LLM in this course (Part 3, Programs 7-8) — just at a toy scale, and this time actually *training* the network ourselves (with backpropagation) instead of only running inference on an already-trained one.

Let's build a tiny corpus where " king"/" queen" and " man"/" woman" always play the same grammatical role (as the subject doing something), while " dog"/" bread"/" country" always play another role (as the object something is done to) — then train a small embedding + a linear layer to predict the next word, and see what structure emerges purely from that.

In [58]:
# Program 6: training a tiny word2vec-style embedding from scratch

import torch.nn as nn

torch.manual_seed(2)  # picked for a clean result -- see the discussion below

# A handful of sentences, repeated many times so the optimizer has enough signal.
sentences = [
    "the king rules the country",
    "the queen rules the country",
    "the man walks the dog",
    "the woman walks the dog",
    "the king is strong",
    "the queen is strong",
    "the man is strong",
    "the woman is strong",
    "the king loves the queen",
    "the man loves the woman",
    "the king eats bread",
    "the queen eats bread",
    "the man eats bread",
    "the woman eats bread",
]
corpus = " ".join(sentences * 30).split()  # simple word-level split, no BPE here

toy_vocab = sorted(set(corpus))
toy_word_to_id = {word: i for i, word in enumerate(toy_vocab)}
print(f"Toy vocabulary: {len(toy_vocab)} words -- {toy_vocab}")

# Build every (current word, next word) pair the corpus contains -- our training data.
training_pairs = [(toy_word_to_id[corpus[i]], toy_word_to_id[corpus[i + 1]]) for i in range(len(corpus) - 1)]
inputs = torch.tensor([pair[0] for pair in training_pairs])
targets = torch.tensor([pair[1] for pair in training_pairs])

# The model: an embedding table (this IS what we're actually training) followed by a
# linear layer that turns an embedding back into a score for every word in the vocabulary.
toy_embedding_dim = 8
toy_embedding = nn.Embedding(len(toy_vocab), toy_embedding_dim)
output_layer = nn.Linear(toy_embedding_dim, len(toy_vocab))

optimizer = torch.optim.Adam(list(toy_embedding.parameters()) + list(output_layer.parameters()), lr=0.03)

for epoch in range(500):
    optimizer.zero_grad()
    predicted_logits = output_layer(toy_embedding(inputs))  # look up embeddings, then predict next word
    loss = F.cross_entropy(predicted_logits, targets)  # how wrong were those predictions?
    loss.backward()  # compute how to adjust every weight, including the embeddings, to do better
    optimizer.step()

print(f"Final training loss: {loss.item():.3f}")

def toy_similarity(word_a, word_b):
    vector_a = toy_embedding.weight[toy_word_to_id[word_a]]
    vector_b = toy_embedding.weight[toy_word_to_id[word_b]]
    return F.cosine_similarity(vector_a.unsqueeze(0), vector_b.unsqueeze(0)).item()

print("\n--- 'subject' words (should end up similar to each other) ---")
for a, b in [("king", "queen"), ("man", "woman"), ("king", "man")]:
    print(f"{a:8} vs {b:8}  {toy_similarity(a, b):.3f}")

print("\n--- 'subject' vs 'object' words (should end up dissimilar) ---")
for a, b in [("king", "dog"), ("king", "country"), ("queen", "bread")]:
    print(f"{a:8} vs {b:8}  {toy_similarity(a, b):.3f}")

print("\n--- 'object' words (should also end up similar to each other) ---")
for a, b in [("dog", "bread"), ("dog", "country")]:
    print(f"{a:8} vs {b:8}  {toy_similarity(a, b):.3f}")

Toy vocabulary: 14 words -- ['bread', 'country', 'dog', 'eats', 'is', 'king', 'loves', 'man', 'queen', 'rules', 'strong', 'the', 'walks', 'woman']
Final training loss: 0.922

--- 'subject' words (should end up similar to each other) ---
king     vs queen     0.674
man      vs woman     0.551
king     vs man       0.635

--- 'subject' vs 'object' words (should end up dissimilar) ---
king     vs dog       -0.362
king     vs country   -0.170
queen    vs bread     0.442

--- 'object' words (should also end up similar to each other) ---
dog      vs bread     0.755
dog      vs country   0.886


Structure emerges exactly where we'd hope: " king"/" queen" (0.674), " man"/" woman" (0.551) and even " king"/" man" (0.635) all end up clearly positive — these are the four words that always play the *subject* role in our sentences. Meanwhile " king" vs " dog" and " king" vs " country" are both *negative*: subjects and objects end up pushed apart. And the objects themselves (" dog", " bread", " country") cluster together too (0.755, 0.886) — sensible, since they're the words a verb like "eats" or "walks" can be followed by. (One pair, " queen"/" bread", comes out only mildly positive rather than clearly negative — a reminder that this is a *toy* example: with just 14 words and one sentence pattern, the separation isn't perfectly clean, but the overall structure is unmistakably there.)

We never told this network anything about grammar, gender, or meaning. All we asked it to do was predict the next word from the previous one — yet to get better at that narrow task, it had no choice but to organize its embedding table so that words playing the same role end up nearby, and words playing different roles end up apart. That's the whole trick behind word2vec, and behind every embedding we've inspected in this notebook: the "meaning" captured in an embedding is a byproduct of training on next-token prediction, at whatever scale — 14 words and 500 training steps here, trillions of words and a full LLM everywhere else.

## Beyond the token: embedding a whole sentence

Token embeddings are useful, but most of the time we want to compare entire sentences, not single words — for example, to find out if two sentences mean roughly the same thing. A simple, homemade way to get a sentence embedding out of any model: run the sentence through it, and average ("mean pool") the resulting per-token vectors into a single one.

In [59]:
# Program 7: a homemade sentence embedding, by mean-pooling token vectors

from transformers import AutoModel

# AutoModel (not AutoModelForCausalLM) gives us direct access to hidden states,
# without the extra layer that predicts next-token logits.
base_model = AutoModel.from_pretrained(model_name)
base_model.eval()

def sentence_embedding(text):
    """A simple sentence embedding: the average of all its tokens' final hidden states."""
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        hidden_states = base_model(**inputs).last_hidden_state  # shape: (1, num_tokens, embedding_dim)
    return hidden_states.mean(dim=1).squeeze(0)  # average over the tokens -> a single vector

sentences = [
    "The cat sat on the mat.",
    "A feline was resting on the rug.",
    "The stock market crashed yesterday.",
]

embeddings = [sentence_embedding(s) for s in sentences]

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        similarity = F.cosine_similarity(embeddings[i].unsqueeze(0), embeddings[j].unsqueeze(0)).item()
        print(f"{similarity:.3f}  {sentences[i]!r}\n       <->  {sentences[j]!r}")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

0.957  'The cat sat on the mat.'
       <->  'A feline was resting on the rug.'
0.746  'The cat sat on the mat.'
       <->  'The stock market crashed yesterday.'
0.793  'A feline was resting on the rug.'
       <->  'The stock market crashed yesterday.'


The two sentences that mean roughly the same thing (the cat/feline ones) score noticeably higher than either has with the unrelated sentence about the stock market — even though they don't share a single word. That's the whole point of a sentence embedding: it captures meaning, not just vocabulary overlap.

## A real embeddings API

Mean-pooling a small local model's hidden states works, but it's a rough approximation — these models weren't specifically trained to produce good sentence embeddings. Providers instead offer dedicated **embedding models**, trained specifically to place similar meanings close together. Like everywhere else in this course, we can call one through the same OpenAI-compatible client, just changing the endpoint: `embeddings.create` instead of `chat.completions.create`. Let's use Gemini's, on the very same three sentences.

In [60]:
# Program 8: sentence embeddings via a real embeddings API (Gemini)

from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

def gemini_embedding(text):
    response = gemini.embeddings.create(model="gemini-embedding-001", input=text)
    return torch.tensor(response.data[0].embedding)

gemini_embeddings = [gemini_embedding(s) for s in sentences]

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        similarity = F.cosine_similarity(gemini_embeddings[i].unsqueeze(0), gemini_embeddings[j].unsqueeze(0)).item()
        print(f"{similarity:.3f}  {sentences[i]!r}\n       <->  {sentences[j]!r}")

0.765  'The cat sat on the mat.'
       <->  'A feline was resting on the rug.'
0.612  'The cat sat on the mat.'
       <->  'The stock market crashed yesterday.'
0.578  'A feline was resting on the rug.'
       <->  'The stock market crashed yesterday.'


Same ranking as our homemade version, but with a clearer gap between the related pair and the unrelated ones — exactly what we'd expect from a model actually trained for this task. Note also the vector length: `gemini-embedding-001` returns 3072 numbers per sentence, regardless of how long the sentence is — a fixed-size summary of its meaning.

## Why this matters for agents

<img src="images/agents.png" width="150" alt="Several agents" style="float: left; margin-right: 15px; margin-bottom: 10px;">

So far, every agent we've built (Part 2's `book_agent`, `recruitment_agent`, and friends) worked by stuffing *everything* it might need — the entire book, every tool description — directly into the system instructions on every single call. That's simple, but it doesn't scale: real documents can be far too large to fit in a single prompt (recall Part 3's context window), and most of that content is irrelevant to any given question anyway.

Embeddings are what makes a better approach possible: instead of sending the whole book every time, you embed it in advance (in chunks), embed the user's question the same way, and use similarity — exactly the cosine similarity we computed above — to find which chunks are actually relevant, and only send *those* to the model. This technique, retrieving relevant content by embedding similarity before generating an answer, is called **RAG** (Retrieval-Augmented Generation), and it's a natural next step for this course.

## Key takeaways

* A token id is just an integer; an **embedding** is the dense vector the model actually reasons with, one per token, stored as a row of a big matrix learned during training.
* Embeddings capture *meaning*, not spelling: tokens (or sentences) that are used similarly end up with similar vectors, measurable with **cosine similarity**.
* The same technique scales from single tokens to whole sentences (via pooling, or a dedicated embeddings API), giving a fixed-size vector that summarizes meaning regardless of length.
* This is the foundation of semantic search and **RAG**: finding relevant content by comparing embeddings, instead of stuffing everything into every prompt.

## Bonus: predicting the words around, not just the next one

Program 5 trained on next-word prediction — a fine pretext task, and the one real LLMs actually use, but not quite the original word2vec recipe. The classic approach (see [Google's Machine Learning Crash Course, "Obtaining embeddings"](https://developers.google.com/machine-learning/crash-course/embeddings/obtaining-embeddings)) instead trains a network to predict the words *around* a given word — every word in a small window on both sides, not just the one right after it. Concretely: a one-hot (or, here, an id) goes in, passes through a hidden layer of size *d* — that hidden layer's weights are exactly the embedding we care about — and the network is trained to predict its neighbors. Once training is done, the classification task itself is thrown away; only that hidden layer's weights get reused, as the embedding table.

Let's train the exact same tiny model as Program 5, but change what it's asked to predict: instead of `(word, next word)` pairs, we'll use every `(word, nearby word)` pair within a window of 2 positions on each side.

In [61]:
# Program 9: word2vec's actual objective -- predicting the surrounding words (skip-gram)

torch.manual_seed(2)

# Same toy sentences as Program 6, rebuilt here so this cell stands on its own.
toy_sentences = [
    "the king rules the country",
    "the queen rules the country",
    "the man walks the dog",
    "the woman walks the dog",
    "the king is strong",
    "the queen is strong",
    "the man is strong",
    "the woman is strong",
    "the king loves the queen",
    "the man loves the woman",
    "the king eats bread",
    "the queen eats bread",
    "the man eats bread",
    "the woman eats bread",
]
skipgram_corpus = " ".join(toy_sentences * 30).split()
skipgram_vocab = sorted(set(skipgram_corpus))
skipgram_word_to_id = {word: i for i, word in enumerate(skipgram_vocab)}

# Skip-gram training pairs: for every word, pair it with every OTHER word within
# `window` positions on either side -- not just the single word right after it.
window = 2
skipgram_pairs = []
for i, center_word in enumerate(skipgram_corpus):
    for j in range(max(0, i - window), min(len(skipgram_corpus), i + window + 1)):
        if j != i:
            skipgram_pairs.append((skipgram_word_to_id[center_word], skipgram_word_to_id[skipgram_corpus[j]]))

print(f"Skip-gram training pairs: {len(skipgram_pairs)} (vs. {len(skipgram_corpus) - 1} for next-word only)")

skipgram_inputs = torch.tensor([pair[0] for pair in skipgram_pairs])
skipgram_targets = torch.tensor([pair[1] for pair in skipgram_pairs])

# Same architecture as Program 6: an embedding table (the hidden layer of size d we
# actually want), followed by a linear layer that scores every word in the vocabulary.
skipgram_embedding = nn.Embedding(len(skipgram_vocab), 8)
skipgram_output_layer = nn.Linear(8, len(skipgram_vocab))
optimizer = torch.optim.Adam(
    list(skipgram_embedding.parameters()) + list(skipgram_output_layer.parameters()), lr=0.03
)

for epoch in range(1500):
    optimizer.zero_grad()
    predicted_logits = skipgram_output_layer(skipgram_embedding(skipgram_inputs))
    loss = F.cross_entropy(predicted_logits, skipgram_targets)
    loss.backward()
    optimizer.step()

print(f"Final training loss: {loss.item():.3f}")

def skipgram_similarity(word_a, word_b):
    vector_a = skipgram_embedding.weight[skipgram_word_to_id[word_a]]
    vector_b = skipgram_embedding.weight[skipgram_word_to_id[word_b]]
    return F.cosine_similarity(vector_a.unsqueeze(0), vector_b.unsqueeze(0)).item()

print("\n--- 'subject' words ---")
for a, b in [("king", "queen"), ("man", "woman")]:
    print(f"{a:8} vs {b:8}  {skipgram_similarity(a, b):.3f}")

print("\n--- 'subject' vs 'object' words ---")
for a, b in [("king", "dog"), ("king", "country"), ("queen", "bread")]:
    print(f"{a:8} vs {b:8}  {skipgram_similarity(a, b):.3f}")

print("\n--- 'object' words ---")
for a, b in [("dog", "bread"), ("dog", "country")]:
    print(f"{a:8} vs {b:8}  {skipgram_similarity(a, b):.3f}")

Skip-gram training pairs: 7434 (vs. 1859 for next-word only)
Final training loss: 1.930

--- 'subject' words ---
king     vs queen     0.673
man      vs woman     0.435

--- 'subject' vs 'object' words ---
king     vs dog       0.068
king     vs country   -0.106
queen    vs bread     -0.085

--- 'object' words ---
dog      vs bread     0.329
dog      vs country   0.352


Same story as Program 5, from a different training signal: " king"/" queen" (0.673) and " man"/" woman" (0.435) end up clearly positive, while " king"/" country" and " queen"/" bread" turn slightly negative — subjects and objects still get pushed apart, even though this time the network never explicitly predicted "the next word", only "some word nearby". The separation is a touch less crisp than Program 5's (this corpus is tiny, and predicting *any* neighbor is a harder, noisier task than predicting *the* next word specifically), but the same phenomenon shows up: a hidden layer trained on nothing but predicting neighboring words spontaneously organizes itself by *role*, exactly as Google's crash course describes.

This is the direct ancestor of what every embedding in this notebook has been doing: word2vec threw away its little classifier once training was done and kept only the hidden layer's weights as a reusable, general-purpose embedding table; a modern LLM's embedding layer is trained the same way in spirit — as a side effect of learning to predict text — except it's trained jointly with the rest of a much larger network (as one giant model, not a separate step), on vastly more data, which is exactly why `SmolLM2`'s real embeddings earlier in this notebook captured relationships (king/queen, the analogy test) far more richly than these 14 toy words ever could.